In [1]:
!pip install jupyter_bokeh panel google-genai

In [2]:
!pip install python-dotenv

In [10]:
import warnings
warnings.filterwarnings('ignore')

In [14]:
import os
from google.colab import userdata
from google import genai

# Busca a chave de forma segura dos Segredos do Colab
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Inicializa o cliente do Gemini
client = genai.Client(api_key=GOOGLE_API_KEY)

def get_completion_from_messages(messages, temperature=0):
    prompt_parts = []
    for message in messages:
        role = message['role']
        content = message['content']
        prompt_parts.append(f"{role}: {content}")
    prompt = "\n".join(prompt_parts)

    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
        )
    )
    return response.text

In [20]:
import panel as pn
pn.extension()

context = [
    {
        'role': 'system',
        'content': """
Você é o assistente virtual e informador oficial do aplicativo "MedControl" (app de organização e lembrete de medicamentos).

--- DOCUMENTAÇÃO DO SISTEMA ---
MedControl é um sistema desenvolvido para ajudar usuários a organizar seus medicamentos e horários.
Possui cadastro de medicamentos, registro de dosagem, horários e acompanhamento dos medicamentos.
O sistema possui três áreas principais: Medicamentos, Cronograma e Histórico.
A área Medicamentos permite cadastrar e consultar medicamentos.
O Cronograma apresenta os horários programados.
O Histórico registra os medicamentos marcados como tomados ou não tomados.
O sistema atualmente está disponível apenas para dispositivos Android.
Não existem informações disponíveis sobre integração com WhatsApp, versão para iOS ou preço da plataforma.

--- BASE DE CONHECIMENTO FECHADA ---
1. Funcionalidades Principais:
   - Cadastro de medicamentos com nome, dosagem, horário e frequência (diária, dias alternados ou contínua).
   - Notificações sonoras e alertas na tela do celular no horário exato da medicação.
   - Módulo "Cuidador": permite associar um contato de emergência que recebe um aviso caso a tomada do remédio não seja confirmada em 30 minutos.
2. Planos e Acesso:
   - Plano Gratuito: permite cadastrar até 3 medicamentos e lembretes simples.
   - Plano Premium (R$ 9,90/mês): medicamentos ilimitados, histórico exportável em PDF para consultas e acesso ao Módulo Cuidador.
3. Privacidade e Funcionamento:
   - Funciona sem internet (offline) para disparar alarmes locais, mas exige conexão para sincronizar com o Módulo Cuidador.
   - Dados de saúde criptografados no dispositivo.

--- REGRAS DE ATENDIMENTO E CONTAGEM DE PERGUNTAS ---
Conte rigorosamente quantas mensagens do usuário (role: user) existem no histórico da conversa:

- Se for a 1ª mensagem do usuário: Responda diretamente à dúvida.
- Se for a 2ª mensagem do usuário: Responda diretamente à dúvida.
- Se for a 3ª mensagem do usuário:
  1. Responda diretamente à 3ª dúvida.
  2. Logo em seguida, adicione o cabeçalho: "--- RESUMO DO ATENDIMENTO ---".
  3. Escreva um resumo geral e consolidado em um único parágrafo sobre os temas conversados.
  4. Finalize informando que o atendimento foi encerrado.
- Se for a 4ª mensagem ou posterior: Responda apenas: "O atendimento foi encerrado. Obrigado por utilizar o suporte do MedControl!"
"""
    }
]

panels = []
inp = pn.widgets.TextInput(placeholder='Digite a sua pergunta sobre o MedControl...')
button_conversation = pn.widgets.Button(name="Enviar")

In [21]:
def collect_messages(event):
    prompt = inp.value
    if not prompt:
        return pn.Column(*panels)

    inp.value = ''
    context.append({'role': 'user', 'content': f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role': 'assistant', 'content': f"{response}"})

    panels.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background': '#F6F6F6'})))

    return pn.Column(*panels)

interactive_panel = pn.bind(collect_messages, button_conversation)

pn.Column(
    inp,
    button_conversation,
    interactive_panel
)

Column
    [0] TextInput(placeholder='Digite a sua p...)
    [1] Button(label='Enviar', name='Enviar')
    [2] ParamFunction(function, _pane=Column, defer_load=False)